In [33]:
import numpy as np
import pandas as pd

In [34]:
# Step 1: Import Data and label and join them
values_df = pd.read_csv('Water_Pump_Training_Set_Values.csv')
labels_df = pd.read_csv('Water_Pump_Training_Set_Labels.csv')


In [35]:
df = values_df.merge(labels_df, on="id", how="inner")

In [36]:
df.shape

(59400, 41)

In [37]:
print(df.columns)

Index(['id', 'amount_tsh', 'date_recorded', 'funder', 'gps_height',
       'installer', 'longitude', 'latitude', 'wpt_name', 'num_private',
       'basin', 'subvillage', 'region', 'region_code', 'district_code', 'lga',
       'ward', 'population', 'public_meeting', 'recorded_by',
       'scheme_management', 'scheme_name', 'permit', 'construction_year',
       'extraction_type', 'extraction_type_group', 'extraction_type_class',
       'management', 'management_group', 'payment', 'payment_type',
       'water_quality', 'quality_group', 'quantity', 'quantity_group',
       'source', 'source_type', 'source_class', 'waterpoint_type',
       'waterpoint_type_group', 'status_group'],
      dtype='object')


In [38]:
# Step 17: Drop columns that are not suitable for ordinary linear regression.
# - recorded_by / num_private: near-constant or no useful signal in this dataset.
# - wpt_name, subvillage, scheme_name, funder, installer: very high cardinality text → huge/unstable one-hot.
# - ward, lga: very many levels; region/basin/region_code already summarize geography.
# - Duplicate “group” columns (same information as the main column, multicollinearity if both one-hot encoded).
cols_to_drop = [
    "funder",
    "installer",
    "wpt_name",
    "num_private",
    "subvillage",
    "region",
    "region_code",
    "district_code",
    "lga",
    "ward",
    "recorded_by",
    "scheme_management",
    "scheme_name",
    "extraction_type",
    "extraction_type_group",
    "management",
    "payment",
    "water_quality",
    "quantity",
    "source",
    "source_type",
    "waterpoint_type"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(df.columns)

Index(['id', 'amount_tsh', 'date_recorded', 'gps_height', 'longitude',
       'latitude', 'basin', 'population', 'public_meeting', 'permit',
       'construction_year', 'extraction_type_class', 'management_group',
       'payment_type', 'quality_group', 'quantity_group', 'source_class',
       'waterpoint_type_group', 'status_group'],
      dtype='object')


In [39]:
# Step 8: Parse date_recorded as datetime (for later features like pump age).
df["date_recorded"] = pd.to_datetime(df["date_recorded"], errors="coerce")

# Step 9: Keep only rows recorded in 2011, 2012, or 2013 as requested.
df = df[df["date_recorded"].dt.year.isin([2011, 2012, 2013])]



In [40]:
# Step 16: Linear regression needs numeric/categorical inputs — not raw datetimes.
# Keep a numeric year from date_recorded, then drop the datetime column.
df["record_year"] = df["date_recorded"].dt.year

In [41]:

# Step 10: Invalid GPS (0, 0) is not in Tanzania — set to NaN.
bad_geo = (df["longitude"] == 0) & (df["latitude"] == 0)
df.loc[bad_geo, ["longitude", "latitude"]] = np.nan


# Step 11: construction_year == 0 means unknown — replace with NaN.
df.loc[df["construction_year"] == 0, "construction_year"] = np.nan

# Step 12: Negative population is invalid.
df.loc[df["population"] < 0, "population"] = np.nan



In [ ]:
#create a new age_of_construction column and drop the older date columns(record_year and construction_year)
df['age_of_construction'] = np.where(
    df['construction_year'].isna(), 
    np.nan, 
    df['record_year'] - df['construction_year'])

df = df.drop(['record_year', 'construction_year', 'date_recorded'], axis=1)

In [48]:
print(df.isnull().sum())


id                       0
amount_tsh               0
date_recorded            0
gps_height               0
longitude                0
latitude                 0
basin                    0
population               0
public_meeting           0
permit                   0
extraction_type_class    0
management_group         0
payment_type             0
quality_group            0
quantity_group           0
source_class             0
waterpoint_type_group    0
status_group             0
age_of_construction      0
dtype: int64


In [45]:
# Only drops the row if 'Price' or 'ID' is NaN
df = df.dropna(subset=['public_meeting', 'permit', 'age_of_construction'])

In [46]:
df.shape

(34688, 19)

In [ ]:
from sklearn.model_selection import train_test_split

# Predictors: drop id (identifier) and raw target. date_recorded only if it still exists (e.g. if you dropped it earlier, this is safe).
_x_drop = [c for c in ("id", "status_group", "date_recorded") if c in df.columns]
X = df.drop(columns=_x_drop)
y = df["status_group"]

# Categorical / boolean columns → will be one-hot encoded for linear regression
categorical_cols = [
    "basin",
    "extraction_type_class",
    "management_group",
    "payment_type",
    "quality_group",
    "quantity_group",
    "source_class",
    "waterpoint_type_group",
    "public_meeting",
    "permit",
]
categorical_cols = [c for c in categorical_cols if c in X.columns]

# 80% train / 20% test; stratify=y keeps similar class proportions in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

In [ ]:
# One-hot encode categorical / boolean features (train vs test, align columns)
X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=False)
X_test_enc = pd.get_dummies(X_test, columns=categorical_cols, drop_first=False)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)
print("X_train_enc:", X_train_enc.shape, "| X_test_enc:", X_test_enc.shape)

In [ ]:
# One-hot encode target (one column per class)
Y_train = pd.get_dummies(y_train, prefix="target").astype(float)
Y_test = pd.get_dummies(y_test, prefix="target").astype(float)
Y_train.columns = [c.replace(" ", "_") for c in Y_train.columns]
Y_test.columns = [c.replace(" ", "_") for c in Y_test.columns]
Y_test = Y_test.reindex(columns=Y_train.columns, fill_value=0)

TARGET_COLS = [
    "target_functional",
    "target_functional_needs_repair",
    "target_non_functional",
]
TARGET_COLS = [c for c in TARGET_COLS if c in Y_train.columns]
print("Y_train:", Y_train.shape, "| target columns:", TARGET_COLS)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Fill NaN with train medians (same features for all three models)
train_medians = X_train_enc.median()
X_train_fit = X_train_enc.fillna(train_medians)
X_test_fit = X_test_enc.fillna(train_medians)

# One LinearRegression per target column (binary 0/1 for that class)
linreg_models = {}
for col in TARGET_COLS:
    model = LinearRegression()
    model.fit(X_train_fit, Y_train[col])
    linreg_models[col] = model

# Test set: separate column for each model's predicted score
test_model_outputs = pd.DataFrame(index=X_test.index)
for col in TARGET_COLS:
    test_model_outputs[f"pred_{col}"] = linreg_models[col].predict(X_test_fit)

# True one-hot on test (same row order as X_test / Y_test)
for col in TARGET_COLS:
    test_model_outputs[f"true_{col}"] = Y_test[col].values

if "id" in df.columns:
    test_model_outputs.insert(0, "id", df.loc[X_test.index, "id"].values)

print("Test-set table (first 10 rows) — pred_* = each model's output; true_* = actual 0/1:\n")
print(test_model_outputs.head(10).to_string())

In [ ]:
# Validation: each model vs its own binary target on TEST
print("Per-model regression metrics on TEST (predicting 0/1 for that class)\n")
for col in TARGET_COLS:
    y_true = Y_test[col].values
    y_hat = test_model_outputs[f"pred_{col}"].values
    mse = mean_squared_error(y_true, y_hat)
    r2 = r2_score(y_true, y_hat)
    print(f"{col}")
    print(f"  MSE: {mse:.4f}  |  R²: {r2:.4f}")

# Optional: one combined class prediction = argmax across the three pred_* scores
suffix_to_label = {
    "functional": "functional",
    "functional_needs_repair": "functional needs repair",
    "non_functional": "non functional",
}
score_cols = [f"pred_{c}" for c in TARGET_COLS]
suffixes = [c.replace("target_", "") for c in TARGET_COLS]
idx = np.argmax(test_model_outputs[score_cols].values, axis=1)
y_test_pred = np.array([suffix_to_label[suffixes[i]] for i in idx])
print("\nAccuracy (test, argmax over the 3 model outputs):", (y_test_pred == y_test.values).mean())